## Datos de caminata
[link](https://archive.ics.uci.edu/dataset/240/human%2Bactivity%2Brecognition%2Busing%2Bsmartphones)

In [250]:
import numpy as np
import pandas as pd
import time

from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import (
    Conv1D,
    MaxPooling1D,
    Flatten,
    Dense,
    Dropout
)

from sklearn.preprocessing import StandardScaler
import joblib

In [251]:
MAPA_ACTIVIDADES = {
    1: "Caminando",
    2: "Caminando hacia arriba",
    3: "Caminando hacia abajo",
    4: "Sentado",
    5: "Parado",
    6: "Acostado"
}

## CARGAR DATASET PRINCIPAL

In [252]:
datos = np.load(
    "datos/har_dataset.npz"
)

X_train = datos["X_train"]
y_train = datos["y_train"]

X_test = datos["X_test"]
y_test = datos["y_test"]

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


# CARGAR DATOS INDIVIDUALES

datos_individual = np.load(
    "datos/har_dataset_individual.npz"
)

X_individual = datos_individual["X_individual"]
y_individual = datos_individual["y_individual"]

indices_individuales = datos_individual[
    "indices_individuales"
]

def nombre_actividad(etiqueta):
    return MAPA_ACTIVIDADES[int(etiqueta)]


print("Datos cargados correctamente:\n")

print(f"X_train:      {X_train.shape} | y_train:      {y_train.shape}")
print(f"X_test:       {X_test.shape} | y_test:       {y_test.shape}")
print(f"X_individual: {X_individual.shape} | y_individual: {y_individual.shape}")

print(f"indices individuales: {indices_individuales}")


print("\nActividades:")

for numero, nombre in MAPA_ACTIVIDADES.items():
    print(f"{numero}: {nombre}")


Datos cargados correctamente:

X_train:      (7352, 561) | y_train:      (7352,)
X_test:       (2947, 561) | y_test:       (2947,)
X_individual: (10, 561) | y_individual: (10,)
indices individuales: [ 253  262  277  593 1273 1290 1924 2053 2274 2526]

Actividades:
1: Caminando
2: Caminando hacia arriba
3: Caminando hacia abajo
4: Sentado
5: Parado
6: Acostado


### FUNCION PARA METRICAS

In [253]:
def calcular_metricas(y_real, y_pred):

    return {
        "accuracy": accuracy_score(y_real, y_pred),
        "precision": precision_score(
            y_real,
            y_pred,
            average="weighted",
            zero_division=0
        ),
        "recall": recall_score(
            y_real,
            y_pred,
            average="weighted",
            zero_division=0
        ),
        "f1_score": f1_score(
            y_real,
            y_pred,
            average="weighted",
            zero_division=0
        )
    }

## MLP

In [254]:
modelo_mlp = MLPClassifier(

    hidden_layer_sizes=(
        128,
        64
    ),

    activation="relu",

    solver="adam",

    max_iter=100,

    early_stopping=True,

    validation_fraction=0.1,

    n_iter_no_change=10,

    random_state=42
)

### Entrenamiento

In [255]:
inicio = time.perf_counter()

modelo_mlp.fit(
    X_train,
    y_train
)

tiempo_entrenamiento_mlp = (time.perf_counter() - inicio)

### Prediccion

In [256]:
inicio = time.perf_counter()

y_pred_mlp = modelo_mlp.predict(X_test)

tiempo_prediccion_mlp = (time.perf_counter() - inicio)

### Metricas

In [257]:
metricas_mlp = calcular_metricas(
    y_test,
    y_pred_mlp
)

matriz_mlp = confusion_matrix(
    y_test,
    y_pred_mlp
)


print("\nResultados MLP:")

for nombre, valor in metricas_mlp.items():
    print(f"{nombre}: {valor:.4f}")

print(f"Tiempo entrenamiento: {tiempo_entrenamiento_mlp:.4f} s")
print(f"Tiempo prediccion: {tiempo_prediccion_mlp:.4f} s")


Resultados MLP:
accuracy: 0.9427
precision: 0.9437
recall: 0.9427
f1_score: 0.9426
Tiempo entrenamiento: 2.6675 s
Tiempo prediccion: 0.0074 s


## CNN

In [258]:
X_train_cnn = X_train_scaled.reshape(
    X_train_scaled.shape[0],
    X_train_scaled.shape[1],
    1
)

X_test_cnn = X_test_scaled.reshape(
    X_test_scaled.shape[0],
    X_test_scaled.shape[1],
    1
)

In [259]:
modelo_cnn =  tf.keras.Sequential([

    tf.keras.layers.Input(shape=(561, 1)),

    tf.keras.layers.Conv1D(
        filters=4,
        kernel_size=11,
        activation="sigmoid"
    ),
    tf.keras.layers.MaxPooling1D(pool_size=4),

    tf.keras.layers.Conv1D(
        filters=8,
        kernel_size=11,
        activation="relu"
    ),
    tf.keras.layers.MaxPooling1D(pool_size=4),

    tf.keras.layers.Flatten(),

    tf.keras.layers.Dense(
        2,
        activation="sigmoid"
    ),

    tf.keras.layers.Dense(
        6,
        activation="softmax"
    )
])


modelo_cnn.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.01),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)


modelo_cnn.summary()

Model: "sequential_11"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d_20 (Conv1D)              │ (None, 551, 4)         │            48 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_19 (MaxPooling1D) │ (None, 137, 4)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_21 (Conv1D)              │ (None, 127, 8)         │           360 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_20 (MaxPooling1D) │ (None, 31, 8)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_11 (Flatten)            │ (None, 248)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_22 (Dense)                │ (None, 2)              │           498 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_23 (Dense)                │ (None, 6)              │            18 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 924 (3.61 KB)

 Trainable params: 924 (3.61 KB)

 Non-trainable params: 0 (0.00 B)

### ENTRENAR

In [260]:
inicio = time.perf_counter()

historial_cnn = modelo_cnn.fit(
    X_train_cnn,
    y_train - 1,
    epochs=5,
    batch_size=256,
    validation_split=0.1,
    verbose=1
)

tiempo_entrenamiento_cnn = time.perf_counter() - inicio

Epoch 1/5
26/26 ━━━━━━━━━━━━━━━━━━━━ 2s 19ms/step - accuracy: 0.1820 - loss: 1.7962 - val_accuracy: 0.1889 - val_loss: 1.7928
Epoch 2/5
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.1917 - loss: 1.7830 - val_accuracy: 0.1889 - val_loss: 1.7942
Epoch 3/5
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.1868 - loss: 1.7832 - val_accuracy: 0.1685 - val_loss: 1.7939
Epoch 4/5
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.1862 - loss: 1.7833 - val_accuracy: 0.1889 - val_loss: 1.7939
Epoch 5/5
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.1917 - loss: 1.7831 - val_accuracy: 0.1889 - val_loss: 1.7941


### PREDICCION

In [261]:
inicio = time.perf_counter()

predicciones = modelo_cnn.predict(
    X_test_cnn,
    verbose=0
)

tiempo_prediccion_cnn = time.perf_counter() - inicio

In [262]:
y_pred_cnn = np.argmax(
    predicciones,
    axis=1
) + 1

### METRICAS

In [263]:
metricas_cnn = calcular_metricas(
    y_test,
    y_pred_cnn
)

matriz_cnn = confusion_matrix(
    y_test,
    y_pred_cnn
)

print("\nResultados CNN:")

for nombre, valor in metricas_cnn.items():
    print(f"{nombre}: {valor:.4f}")

print(f"Tiempo entrenamiento: {tiempo_entrenamiento_cnn:.4f} s")
print(f"Tiempo prediccion: {tiempo_prediccion_cnn:.4f} s")


Resultados CNN:
accuracy: 0.1822
precision: 0.0332
recall: 0.1822
f1_score: 0.0562
Tiempo entrenamiento: 2.7341 s
Tiempo prediccion: 0.7271 s


## RESULTADOS GENERALES

In [264]:
resultados_generales_1 = pd.DataFrame([
    {
        "modelo": "MLP",

        "accuracy": metricas_mlp["accuracy"],
        "precision": metricas_mlp["precision"],
        "recall": metricas_mlp["recall"],
        "f1_score": metricas_mlp["f1_score"],

        "tiempo_entrenamiento_s": tiempo_entrenamiento_mlp,
        "tiempo_prediccion_s": tiempo_prediccion_mlp,
        "iteraciones": modelo_mlp.n_iter_
    },

    {
        "modelo": "CNN",

        "accuracy": metricas_cnn["accuracy"],
        "precision": metricas_cnn["precision"],
        "recall": metricas_cnn["recall"],
        "f1_score": metricas_cnn["f1_score"],

        "tiempo_entrenamiento_s": tiempo_entrenamiento_cnn,
        "tiempo_prediccion_s": tiempo_prediccion_cnn,
        "iteraciones": len(historial_cnn.history["loss"])
    }
])

In [265]:
print(resultados_generales_1.to_string(index=False))

print("\nMatriz de confusión MLP:")
print(matriz_mlp)

print("\nMatriz de confusión CNN:")
print(matriz_cnn)

modelo  accuracy  precision   recall  f1_score  tiempo_entrenamiento_s  tiempo_prediccion_s  iteraciones
   MLP  0.942654   0.943727 0.942654  0.942601                2.667469             0.007447           32
   CNN  0.182219   0.033204 0.182219  0.056172                2.734081             0.727141            5

Matriz de confusión MLP:
[[483   1  12   0   0   0]
 [ 25 443   2   0   1   0]
 [  6  27 387   0   0   0]
 [  0   3   0 462  24   2]
 [  0   0   0  66 466   0]
 [  0   0   0   0   0 537]]

Matriz de confusión CNN:
[[  0   0   0   0   0 496]
 [  0   0   0   0   0 471]
 [  0   0   0   0   0 420]
 [  0   0   0   0   0 491]
 [  0   0   0   0   0 532]
 [  0   0   0   0   0 537]]


## PRUEBA INDIVIDUAL

In [266]:
X_individual_cnn = X_individual.reshape(
    -1,
    561,
    1
)

In [267]:
resultados_individuales = []

for i in range(len(X_individual)):

    etiqueta_real = int(y_individual[i])
    actividad_real = (nombre_actividad(etiqueta_real))


    # ==================
    # MLP

    muestra_mlp = (X_individual[i:i+1])

    inicio = time.perf_counter()

    pred_mlp = int(modelo_mlp.predict(muestra_mlp)[0])

    tiempo_mlp = (time.perf_counter() - inicio)

    actividad_mlp = (nombre_actividad(pred_mlp))


    # ======================================
    # CNN

    muestra_cnn = (X_individual_cnn[i:i+1])

    inicio = time.perf_counter()

    probabilidades = (
        modelo_cnn.predict(
            muestra_cnn,
            verbose=0
        )[0]
    )

    tiempo_cnn = (time.perf_counter() - inicio)

    pred_cnn = (int(np.argmax(probabilidades)) + 1)

    actividad_cnn = (nombre_actividad(pred_cnn))

    confianza_cnn = float(probabilidades[pred_cnn - 1])

    # GUARDAR
    
    resultados_individuales.append({
        "indice_original": int(indices_individuales[i]),

        # REAL
        "etiqueta_real": etiqueta_real,
        "actividad_real": actividad_real,

        # MLP
        "prediccion_MLP": pred_mlp,
        "actividad_MLP": actividad_mlp,
        "correcto_MLP": pred_mlp == etiqueta_real,
        "tiempo_prediccion_MLP_s": tiempo_mlp,

        # CNN
        "prediccion_CNN": pred_cnn,
        "actividad_CNN": actividad_cnn,
        "confianza_CNN": confianza_cnn,
        "correcto_CNN": pred_cnn == etiqueta_real,
        "tiempo_prediccion_CNN_s": tiempo_cnn
    })


In [268]:
resultados_individuales_2 = pd.DataFrame(resultados_individuales)

## GUARDAR RESULTADOS

In [269]:
resultados_generales_1.to_csv("resultados/resultados_generales_2.csv", index=False)

resultados_individuales_2.to_csv("resultados/resultados_individuales_2.csv",index=False)

print("\nResultados guardados correctamente.")


Resultados guardados correctamente.


In [270]:
modelo_cnn.save("modelos/test-2_cnn.keras") 
joblib.dump(modelo_mlp, "modelos/test-2_mlp.joblib")

['modelos/test-2_mlp.joblib']